### Day 09 · 模块与高级特性

**前置**: Day01-08 全部语法点(print/input/变量/字符串/分支/循环/函数/列表字典/文件 I/O/异常/OOP 基础)  
**关键问题**: 代码破千行后,单文件不堪重负 —— 怎么把**拆分到多个文件**、让重复的"计时/日志/资源管理"逻辑自动复用?  
**本日目标**: 掌握模块 import / 生成器 yield / 上下文管理器 __enter__ __exit__ / 装饰器 @decorator,每个知识点后紧跟练习

#### 知识点 1:模块 import / from / as / 包 __init__.py

**痛点**: 一个 .py 写到 1000 行,翻来翻去找函数非常痛苦

**类比**: 模块就像"抽屉"——把不同功能的工具分门别类放进各自的抽屉,用到哪个拉开哪个,不需要把全堆在桌面上

**核心概念**:
- `import 模块名` → 导入整个模块,用 `模块名.成员` 访问
- `from 模块名 import 成员` → 直接把成员拿过来用
- `from 模块名 import 成员 as 别名` → 改名,防止命名冲突
- 自定义模块:任意 .py 文件都可以被 import
- 包(package):带 `__init__.py` 的文件夹,可包含多个模块
- `__all__`:控制 `from 包 import *` 时能导出哪些成员

**import 模块(完整路径)**

最基础的导入方式。把整个模块引入当前命名空间后,通过 `模块名.成员名` 访问其中的函数/常量/类。优点是不会和当前文件的名字冲突,缺点是多打几个字。

In [ ]:
# 导入 Python 标准库中的 math 模块
import math

# 通过 模块名.成员名 的方式调用 sqrt
result = math.sqrt(16)
print(result)               # 4.0

# 也可以访问模块中的常量,例如圆周率
print(math.pi)              # 3.141592653589793


**from ... import(直接使用)**

把模块中的**某个成员**直接引入当前命名空间后,调用时**不需要**再写模块前缀。注意如果当前文件已有同名成员会被覆盖,要谨慎。

In [ ]:
# 从 math 模块中只引入 sqrt 这一个函数
from math import sqrt

# 调用时直接写函数名,不再需要 math. 前缀
print(sqrt(25))             # 5.0
print(sqrt(2))              # 1.4142135623730951


**import ... as(别名)**

当模块名太长、或与现有名字冲突时,用 `as` 给成员起一个更短的别名。这是团队协作中常见的约定,例如 `import numpy as np`。

In [ ]:
# 把 math 模块中的 factorial 命名为别名 fac
from math import factorial as fac

# 调用时使用别名 fac,效果完全等同于 factorial
print(fac(5))               # 120
print(fac(0))               # 1


**自定义模块与包结构**

把代码放进一个 `.py` 文件就是**模块**,同目录直接 `import` 即可使用。多个模块放进带 `__init__.py` 的文件夹里就构成**包**(package)。`__all__` 列表控制 `from 包 import *` 时能导出哪些名字。

In [ ]:
# === 自定义模块 ===
# 文件 utils.py 内容:
#   def greet(name):
#       return f"你好, {name}"
#
# 同目录下直接 import
# import utils
# print(utils.greet("小明"))

# === 包结构 ===
# mypack/
#   __init__.py   <-- 必须有,可以是空文件
#   math_tools.py
#   string_tools.py
#
# __init__.py 中写:
#   __all__ = ["math_tools", "string_tools"]
#
# 使用:
# from mypack import math_tools
# math_tools.circle_area(5)

# __all__ 的作用:控制 from mypack import * 暴露哪些模块
# 没在 __all__ 里的模块不会被 * 导入
print("模块/包 概念演示完成")


** ---- 执行过程 ----**

以 `from math import sqrt` 为例:

1. Python 在 sys.path 列表里找到 math 模块
2. 执行 math 模块的代码,在内存中创建一个模块对象
3. 从模块对象中取出 sqrt 这个名字
4. 把 sqrt 绑定到当前文件的命名空间
5. 之后调用 sqrt() 时直接用当前名字,不再找 math

导入只执行一次,重复 import 不会重复执行模块代码。

**逐行解剖** — 下面这段代码把四种导入方式集中演示,注意每一行的写法差异和对命名空间的影响:

In [ ]:
import math
from math import sqrt
from math import factorial as fac

print(math.sqrt(9))         # 通过模块名访问
print(sqrt(9))              # 直接访问
print(fac(4))               # 通过别名访问,等于 factorial(4)


**练习 1** — 写一个 `math_tools.py` 模块,里面放一个 `circle_area(radius)` 函数,返回圆的面积(使用 3.14159)。然后在主代码中导入它,计算半径为 5 的面积。

```python
# math_tools.py
def circle_area(radius):
    return 3.14159 * radius * radius
```

> 问自己:如果 math_tools.py 不在当前目录能 import 吗?怎么让 Python 找到子目录里的模块?

In [ ]:
# ==============================
# 学员代码区:导入 math_tools 并调用
# ==============================
pass


In [ ]:
# 参考答案
# 假设 math_tools.py 与本文件同目录
# import math_tools
# area = math_tools.circle_area(5)
# print(area)                # 78.53975

# 或
# from math_tools import circle_area
# print(circle_area(5))
print("参考答案完成")


**练习 2** — 创建如下包结构,在 `__init__.py` 中用 `__all__` 导出两个子模块:

```
mypack/
  __init__.py
  math_tools.py      # 放 circle_area()
  string_tools.py    # 放 reverse(s)
```

`__init__.py` 中写:
```python
__all__ = ["math_tools", "string_tools"]
```

> 问自己:如果 `__init__.py` 文件夹是空的,Python 还把它当作包吗(Python 3.3+ 隐式命名空间包)?显式写 `__init__.py` 有什么好处?

In [ ]:
# ==============================
# 学员代码区:搭建包结构
# ==============================
pass


In [ ]:
# 参考答案:mypack/__init__.py
# __all__ = ["math_tools", "string_tools"]
#
# mypack/math_tools.py:
#   def circle_area(radius):
#       return 3.14159 * radius * radius
#
# mypack/string_tools.py:
#   def reverse(s):
#       return s[::-1]
#
# 使用:
# from mypack import math_tools
# print(math_tools.circle_area(5))
print("参考答案完成")


#### 知识点 2:生成器 yield / next / yield from

**痛点**: 有 100 万个数据要处理,一次性读到列表里会撑爆内存

**类比**: 生成器就像"按需点餐"——你吃一盘上一盘,不像列表那样一次性把菜全端上来( buffet )

**核心概念**:
- `yield`:暂停函数并返回一个值,下次从暂停处继续
- 包含 `yield` 的函数就是**生成器函数**,调用它返回生成器对象
- `next(gen)`:手动推进生成器,产出一个值
- `yield from`:把迭代工作委托给另一个可迭代对象
- 生成器**一次性**,遍历完就没了,不能 len(),不能索引
- 生成器表达式 `(x for x in range(n))` 写法像列表推导但用圆括号

**生成器 yield 基础**

普通函数 `return` 一次就彻底结束,而 `yield` 可以**暂停并产出一个值**,下次调用时从暂停处继续执行。这种"暂停/继续"能力由 Python 自动实现。

In [ ]:
def squares(n):
    # 生成器函数:用 yield 逐个产出平方数
    for i in range(n):
        yield i * i

# 调用生成器函数不会执行函数体,而是返回生成器对象
gen = squares(4)
print(type(gen))            # <class 'generator'>

# 用 for 循环一次性遍历
gen2 = squares(4)
for v in gen2:
    print(v)                # 0 1 4 9


**生成器的特性:一次性与 yield from**

生成器**只能遍历一次**,再次遍历为空。`yield from` 可把工作委托给另一个可迭代对象,等价于 `for item in iterable: yield item`。

In [ ]:
def squares(n):
    for i in range(n):
        yield i * i

# 一次性:第二次遍历为空
gen = squares(3)
print(list(gen))            # [0, 1, 4]
print(list(gen))            # []

# yield from 委托
def chain(*iterables):
    for it in iterables:
        yield from it       # 等价于 for x in it: yield x

print(list(chain([1, 2], [3, 4])))   # [1, 2, 3, 4]


**生成器与列表的取舍**

列表把全部数据存进内存,支持 `len()` 和多次遍历;生成器逐个产出,**省内存但只遍历一次,且没有长度**。数据量大时用生成器,数据小需要反复用时用列表。

In [ ]:
import sys

# 列表占内存大,生成器几乎不占
list_data = [i * i for i in range(1000)]
gen_data = (i * i for i in range(1000))
print(sys.getsizeof(list_data))     # 8856
print(sys.getsizeof(gen_data))      # 200

# 列表支持 len(),生成器不支持
print(len(list_data))               # 1000
# len(gen_data)  # 报错:TypeError


**斐波那契生成器**

斐波那契数列无限长,不可能用列表全部存下来。生成器可以**按需产出**,理论上能无限生成下去。这是"惰性计算"最直接的价值体现。

In [ ]:
def fib(n):
    # 生成前 n 个斐波那契数
    a, b = 0, 1              # 前两个初始值
    for _ in range(n):
        yield a              # 产出当前值
        a, b = b, a + b      # 同步更新 a, b

print(list(fib(10)))         # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

# 用 next() 手动推进
gen = fib(5)
print(next(gen))             # 0
print(next(gen))             # 1
print(next(gen))             # 1


**生成器表达式**

生成器也有表达式写法,语法和列表提导很像,只是把方括号 `[]` 换成圆括号 `()`。在需要**节省内存**时特别有用,比如作为函数参数直接喂给 sum / max。

In [ ]:
# 列表提导:一次性算出所有值,存入内存
squares_list = [x * x for x in range(10)]
print(squares_list)          # [0, 1, 4, 81]

# 生成器表达式:逐个产出,不一次性存
squares_gen = (x * x for x in range(10))
print(squares_gen)           # <generator object ...>

# 直接交给 sum,不需要中间列表
total = sum(x * x for x in range(10))
print(total)                 # 285


** ---- 执行过程 ----**

以 `squares(3)` 为例:

1. 调用 `squares(3)` → 创建生成器对象 gen,**不执行**函数体
2. 第一次 `next(gen)` → 进入循环 i=0,`yield 0` 暂停,返回 0
3. 第二次 `next(gen)` → 从暂停处继续,i=1,`yield 1` 暂停,返回 1
4. 第三次 `next(gen)` → i=2,`yield 4` 暂停,返回 4
5. 第四次 `next(gen)` → for 循环耗尽,抛出 `StopIteration`

for 循环内部自动处理 StopIteration,所以不会报错。

**逐行解剖** — 下面用 yield 实现一个简单的计数器生成器,每一行都标注了执行到时的状态:

In [ ]:
def counter(start=0):
    # 无限生成 start, start+1, start+2, ...
    n = start
    while True:              # 无限循环,每次 yield 暂停
        yield n
        n += 1

c = counter(10)
print(next(c))               # 10
print(next(c))               # 11
print(next(c))               # 12


**练习 1** — 写一个 `read_large_file(path)` 生成器,每次 `yield` 一行内容(去掉末尾换行符)。用生成器读大文件不会一次性把整个文件载入内存。

> 问自己:为什么不直接 `return f.readlines()` ?内存占用差多少?

In [ ]:
# ==============================
# 学员代码区:read_large_file
# ==============================
# def read_large_file(path):
#     ...
pass


In [ ]:
# 参考答案
# def read_large_file(path):
#     # 每次只读一行,内存占用恒定
#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             yield line.rstrip("\n")
#
# 使用示例:
# for line in read_large_file("data.txt"):
#     print(line)
print("参考答案完成")


**练习 2** — 写 `fib_gen(n)` 生成器,产出前 n 个斐波那契数。再写一版普通函数返回列表,对比两者内存占用。

> 问自己:生成器能不能直接 `gen[0]` 取第一个元素?为什么?

In [ ]:
# ==============================
# 学员代码区:fib_gen
# ==============================
# def fib_gen(n):
#     ...
pass


In [ ]:
# 参考答案
# def fib_gen(n):
#     a, b = 0, 1
#     for _ in range(n):
#         yield a
#         a, b = b, a + b
#
# import sys
# print(sys.getsizeof(list(fib_gen(1000))))   # 列表版
# print(sys.getsizeof(fib_gen(1000)))         # 生成器版(几乎不变)
print("参考答案完成")


#### 知识点 3:上下文管理器 __enter__/__exit__

**痛点**: 打开文件/数据库连接后忘记关闭,资源泄漏

**类比**: 上下文管理器就像"自动关门的房间"——进入时自动开灯,离开时自动关灯,哪怕中途发生异常

**核心概念**:
- `with 表达式 as 变量:` → 进入时调用 `__enter__`,退出时调用 `__exit__`
- `__enter__(self)`:进入代码块前执行,返回值赋给 `as` 后的变量
- `__exit__(self, exc_type, exc_val, exc_tb)`:退出时执行,三个参数是异常信息
- `__exit__` 返回 `True` 表示异常已处理,不再向上抛
- `@contextmanager`:用生成器简化上下文管理器写法

**上下文管理器 __enter__ / __exit__**

`with` 语句需要对象实现两个方法:**__enter__**(进入时自动调用)和 **__exit__**(退出时自动调用,无论是否发生异常)。常见场景:文件 `with open(...) as f` 最怕忘记关闭。

In [ ]:
import time

class Timer:
    # 自定义上下文管理器:自动计时

    def __enter__(self):
        # 进入 with 块前执行
        self.start = time.time()
        print("[Timer] 开始计时")
        return self          # 返回值赋给 as 后的变量

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 退出 with 块时执行(异常也会触发)
        elapsed = time.time() - self.start
        print(f"[Timer] 耗时 {elapsed:.4f} 秒")
        # 不返回 True,如有异常照常向上抛

with Timer() as t:
    total = sum(range(1000000))
    print(f"total = {total}")


**__exit__ 的异常处理参数**

`__exit__` 必须接受三个参数:**exc_type**(异常类型)、**exc_val**(异常值)、**exc_tb**(追踪对象)。如果没有异常发生,三个参数都是 `None`。返回 `True` 表示异常已处理,不再向上抛。

In [ ]:
class SafeDivide:
    # 演示 __exit__ 中如何处理异常

    def __enter__(self):
        print("[SafeDivide] 进入")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            print("[SafeDivide] 正常退出")
        else:
            print(f"[SafeDivide] 捕获异常: {exc_type.__name__}: {exc_val}")
            return True     # 吞掉异常,不再向上抛

with SafeDivide():
    result = 10 / 0         # 除零异常,但被吞掉
print("继续执行,没崩溃")


**contextlib.contextmanager 装饰器**

除了手写 `__enter__`/`__exit__`,Python 提供了 `@contextmanager`,用生成器函数简化写法:yield 之前的代码等价于 `__enter__`,yield 之后的代码等价于 `__exit__`。

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer():
    start = time.time()      # 等价于 __enter__
    print("[timer] 开始")
    try:
        yield                # 产出,把控制权交给 with 块
    finally:
        # 等价于 __exit__,无论是否异常都执行
        elapsed = time.time() - start
        print(f"[timer] 耗时 {elapsed:.4f} 秒")

with timer():
    s = sum(range(1000000))
    print(f"s = {s}")


** ---- 执行过程 ----**

以 `with Timer() as t:` 为例:

1. `Timer()` → 创建实例
2. 调用 `实例.__enter__()` → 记录开始时间,返回 self 赋给 t
3. 执行 with 块内部代码
4. 无论正常退出还是异常,调用 `实例.__exit__(...)`
5. `__exit__` 计算耗时并打印

**逐行解剖** — 下面代码展示了一个最常见的错误:用 try/finally 手动关闭 vs 上下文管理器的对比

In [ ]:
# try/finally 手动版(繁琐,容易漏写 finally)
f = open("_tmp.txt", "w", encoding="utf-8")
try:
    f.write("hello")
finally:
    f.close()                # 无论是否异常都关闭

# 上下文管理器版(推荐)
with open("_tmp.txt", "w", encoding="utf-8") as f:
    f.write("hello")        # with 结束自动关闭,不需要 finally

# 清理临时文件
import os
if os.path.exists("_tmp.txt"):
    os.remove("_tmp.txt")
print("ok")


**练习** — 写一个 `ResourceManager` 类作为上下文管理器,模拟打开/关闭一个数据库连接。`__enter__` 时打印"连接已打开"并返回 self,`__exit__` 时打印"连接已关闭"。在 with 块中模拟一次查询操作。

> 问自己:如果 with 块里发生异常,不返回 True 的话异常会怎么传递?

In [ ]:
# ==============================
# 学员代码区:ResourceManager
# ==============================
# class ResourceManager:
#     ...
pass


In [ ]:
# 参考答案
# class ResourceManager:
#     def __init__(self, name):
#         self.name = name
#
#     def __enter__(self):
#         print(f"[DB] {self.name} 连接已打开")
#         return self
#
#     def __exit__(self, exc_type, exc_val, exc_tb):
#         print(f"[DB] {self.name} 连接已关闭")
#         # 不返回 True,异常继续向上抛
#
# with ResourceManager("users") as db:
#     print("执行查询...")
print("参考答案完成")


#### 知识点 4:装饰器 @timer / @functools.wraps

**痛点**: 想给多个函数加"计时"逻辑,每个函数手动改又是复制粘贴

**类比**: 装饰器就像"包装礼物"——不改变礼物本身,在外面多包一层精美的包装纸(附加功能)

**核心概念**:
- 装饰器本质是接收函数/返回函数的高阶函数
- `@functools.wraps(fn)`:保留原函数的 `__name__` / `__doc__`
- `*args, **kwargs`:让装饰器适配任意参数签名
- 带参数的装饰器:三层嵌套(外层接收参数,中层接收函数,内层是包装)
- 多个装饰器叠加:执行顺序**从下往上**(靠近函数的先生效)

**装饰器基础 @timer**

装饰器本质是**接收函数、返回新函数**的高阶函数,语法糖 `@decorator` 让写法简洁。核心思想:**在不修改原函数代码的前提下增加额外功能**。

In [ ]:
import time
import functools

def timer(fn):
    # 装饰器:给函数加计时,不修改原函数
    @functools.wraps(fn)     # 保留原函数的 __name__ 等元信息
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)   # 调用原函数
        elapsed = time.time() - start
        print(f"{fn.__name__} 耗时 {elapsed:.4f} 秒")
        return result
    return wrapper

@timer
def slow_sum(n):
    total = 0
    for i in range(n):
        total += i
    return total

print(slow_sum(1000000))


**functools.wraps 的重要性**

装饰器会**替换**原函数的元信息(函数名、文档字符串等),被装饰后 `__name__` 会变成 `wrapper`。用 `@functools.wraps(fn)` 可以保留这些元信息,对调试很重要。

In [ ]:
import functools

def my_decorator(fn):
    @functools.wraps(fn)     # 关键:保留元信息
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    return wrapper

@my_decorator
def greet(name):
    """打招呼"""
    return f"你好, {name}"

print(greet.__name__)        # greet (没有 @wraps 会输出 wrapper)
print(greet.__doc__)         # 打招呼


**带参数的装饰器与多个装饰器叠加**

当装饰器本身需要接收参数时,需要**三层嵌套**:外层接收参数,中层接收函数,内层是真正的包装逻辑。一个函数可以叠加**多个装饰器**,执行顺序**从下往上**。

In [ ]:
import functools

# === 带参数的装饰器 ===
def log(level="INFO"):
    # 外层:接收装饰器的参数 level
    def decorator(fn):
        # 中层:接收被装饰的函数 fn
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            # 内层:真正的包装逻辑
            print(f"[{level}] 调用 {fn.__name__}")
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@log("DEBUG")
def add(a, b):
    return a + b

print(add(2, 3))             # [DEBUG] 调用 add → 5


In [ ]:
# === 多个装饰器叠加 ===
def deco1(fn):
    # 第一个装饰器
    def wrapper(*args, **kwargs):
        print("deco1 前")
        result = fn(*args, **kwargs)
        print("deco1 后")
        return result
    return wrapper

def deco2(fn):
    # 第二个装饰器
    def wrapper(*args, **kwargs):
        print("deco2 前")
        result = fn(*args, **kwargs)
        print("deco2 后")
        return result
    return wrapper

# 执行顺序:从下往上 → deco2 先,deco1 后
@deco1
@deco2
def hello():
    print("hello 函数")

hello()
# 输出顺序:deco1 前 → deco2 前 → hello → deco2 后 → deco1 后


** ---- 执行过程 ----**

以 `@timer` 装饰 `slow_sum` 为例:

1. 定义 slow_sum 时,`@timer` 等价于 `slow_sum = timer(slow_sum)`
2. 调用 `timer(slow_sum)` → 返回 wrapper 函数
3. 此后 `slow_sum` 这个名字指向的是 wrapper
4. 调用 `slow_sum(1000000)` 时实际执行 wrapper(1000000)
5. wrapper 计时 → 调用原 slow_sum → 再计时 → 返回结果

**逐行解剖** — 下面代码定义一个通用装饰器,展示 `*args, **kwargs` 如何让装饰器适配**任意**参数签名:

In [ ]:
import functools

def universal(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        # *args 打包位置参数为元组
        # **kwargs 打包关键字参数为字典
        print(f"参数: args={args}, kwargs={kwargs}")
        return fn(*args, **kwargs)
    return wrapper

@universal
def f(a, b, c=0):
    return a + b + c

print(f(1, 2))               # args=(1, 2), kwargs={}
print(f(1, 2, c=3))          # args=(1, 2), kwargs={"c": 3}


**练习 1** — 写一个 `@log` 装饰器,每次调用时打印函数名和参数(位置参数 + 关键字参数),然后执行原函数并返回结果。用 `@functools.wraps(fn)` 保留元信息。

> 问自己:如果 wrapper 的参数签名写死(比如 `def wrapper(a, b)`) ,会出什么问题?

In [ ]:
# ==============================
# 学员代码区:@log 装饰器
# ==============================
# import functools
# def log(fn):
#     ...
pass


In [ ]:
# 参考答案
# import functools
#
# def log(fn):
#     @functools.wraps(fn)
#     def wrapper(*args, **kwargs):
#         print(f"调用 {fn.__name__}, args={args}, kwargs={kwargs}")
#         return fn(*args, **kwargs)
#     return wrapper
#
# @log
# def add(a, b):
#     return a + b
#
# print(add(2, 3))
print("参考答案完成")


**练习 2** — 写一个带参数的 `@retry(times=3)` 装饰器,当函数抛出异常时自动重试,共尝试 times 次,如果全部失败则抛最后一次异常。这就是"三层嵌套"的典型应用。

> 问自己:如果异常类型不想全部捕获,只想捕获 ValueError,应该怎么改?

In [ ]:
# ==============================
# 学员代码区:@retry 装饰器
# ==============================
# import functools
# def retry(times=3):
#     ...
pass


In [ ]:
# 参考答案
# import functools
#
# def retry(times=3):
#     def decorator(fn):
#         @functools.wraps(fn)
#         def wrapper(*args, **kwargs):
#             last_err = None
#             for attempt in range(1, times + 1):
#                 try:
#                     return fn(*args, **kwargs)
#                 except Exception as e:
#                     last_err = e
#                     print(f"第 {attempt} 次失败: {e}")
#             raise last_err
#         return wrapper
#     return decorator
print("参考答案完成")


### 今日小结

- **模块与包**: `import 模块`(带前缀)、`from 模块 import 成员`(直接用)、`as` 别名、`__all__` 控制导出
- **生成器**: `yield` 暂停/继续、一次性、省内存、`yield from` 委托
- **上下文管理器**: `__enter__`/`__exit__`、`with` 自动关闭资源、`@contextmanager` 简化写法
- **装饰器**: 高阶函数、`@wraps` 保留元信息、三层嵌套带参数、多装饰器从下往上

四个概念的核心共同点是**复用**:模块复用代码,生成器复用内存,上下文管理器复用资源管理,装饰器复用横切逻辑。

### 更多练习

- 当堂练:course/lesson09/in_class/practice01.py ~ practice06.py
- 课后作业:course/lesson09/homework/task01.py ~ task03.py